# Benchmark do givp em R

Notebook de benchmark da implementacao R (`r/`).
Inclui funcoes classicas de otimizacao para comparacao.

## 1. Setup

In [ ]:
suppressPackageStartupMessages({
  library(devtools)
  library(givp)
  library(dplyr)
})

set.seed(42)
cat("Pacotes carregados com sucesso.\n")


## 2. Helpers

In [ ]:
default_config <- function(seed = 42L) {
  givp::givp_config(
    max_iterations = 60L,
    alpha = 0.15,
    vnd_iterations = 30L,
    ils_iterations = 8L,
    perturbation_strength = 4L,
    num_candidates_per_step = 15L,
    elite_size = 5L,
    path_relink_frequency = 10L,
    early_stop_threshold = 40L,
    adaptive_alpha = TRUE,
    seed = seed
  )
}

solve_n_runs <- function(
  fn, bounds, n_runs = 3L, direction = "minimize"
) {
  vals <- numeric(n_runs)
  times <- numeric(n_runs)
  for (i in seq_len(n_runs)) {
    cfg <- default_config(seed = i - 1L)
    t0 <- Sys.time()
    res <- givp::givp(
      func = fn,
      bounds = bounds,
      direction = direction,
      config = cfg
    )
    elapsed <- difftime(Sys.time(), t0, units = "secs")
    times[i] <- as.numeric(elapsed)
    vals[i] <- res$fun
  }
  list(values = vals, times = times)
}


## 3. Funcoes benchmark

In [ ]:
sphere <- function(x) sum(x^2)

rosenbrock <- function(x) {
  sum(100 * (x[-1] - x[-length(x)]^2)^2 + (1 - x[-length(x)])^2)
}

rastrigin <- function(x) {
  10 * length(x) + sum(x^2 - 10 * cos(2 * pi * x))
}

ackley <- function(x) {
  n <- length(x)
  -20 * exp(-0.2 * sqrt(sum(x^2) / n)) -
    exp(sum(cos(2 * pi * x)) / n) + 20 + exp(1)
}

benchmarks <- list(
  Sphere = list(
    fn = sphere,
    bounds = replicate(10, c(-5.12, 5.12), simplify = FALSE),
    optimum = 0
  ),
  Rosenbrock = list(
    fn = rosenbrock,
    bounds = replicate(5, c(-2.048, 2.048), simplify = FALSE),
    optimum = 0
  ),
  Rastrigin = list(
    fn = rastrigin,
    bounds = replicate(10, c(-5.12, 5.12), simplify = FALSE),
    optimum = 0
  ),
  Ackley = list(
    fn = ackley,
    bounds = replicate(10, c(-32.768, 32.768), simplify = FALSE),
    optimum = 0
  )
)


## 4. Executar benchmark

In [ ]:
results <- lapply(names(benchmarks), function(name) {
  b <- benchmarks[[name]]
  run <- solve_n_runs(b$fn, b$bounds, n_runs = 3L)
  best <- min(run$values)
  err_abs <- abs(best - b$optimum)
  err_rel <- 100 * err_abs / max(abs(b$optimum), 1.0)
  data.frame(
    Problem = name,
    Best = best,
    Mean = mean(run$values),
    SD = sd(run$values),
    ErrorAbs = err_abs,
    ErrorRelPct = err_rel,
    MeanTimeS = mean(run$times)
  )
})

df <- bind_rows(results)
print(df)


## 5. Plot simples

In [ ]:
barplot(
    df$ErrorRelPct,
    names.arg = df$Problem,
    las = 2,
    col = "steelblue",
    ylab = "Erro relativo (%)"
)
